In [7]:
import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("building_health_monitoring_dataset.csv")

La columna objetivo es Condition Label, y originalmente el dataset marca 3 grados de daño:

0 $\rightarrow$ Daño nulo

1 $\rightarrow$ Ligero Daño

2 $\rightarrow$ Mayor Daño

Como queremos usar una regresión logística, es mejor separarlo en 2 posibles grados, dañado y no dañado, por lo que:

0 $\rightarrow$ No Dañado $\rightarrow$ 0

1 $\rightarrow$ Dañado $\rightarrow$ 1

2 $\rightarrow$ Dañado $\rightarrow$ 1

In [14]:
df.columns

Index(['Timestamp', 'Accel_X (m/s^2)', 'Accel_Y (m/s^2)', 'Accel_Z (m/s^2)',
       'Strain (με)', 'Temp (°C)', 'Condition Label', 'Damage'],
      dtype='str')

In [ ]:
df["Damage"] = (df["Condition Label"] > 0).astype(int)

,Timestamp,Accel_X (m/s^2),Accel_Y (m/s^2),Accel_Z (m/s^2),Strain (με),Temp (°C),Condition Label,Damage
0,2025-04-19 00:00:00,0.149014,0.419807,9.742482,61.843849,23.704760,0,0
1,2025-04-19 00:00:01,-0.041479,0.277390,9.795548,82.792300,24.953195,0,0
2,2025-04-19 00:00:02,0.194307,0.017889,9.730758,91.727889,25.027025,0,0
3,2025-04-19 00:00:03,0.456909,-0.194081,9.779204,137.753753,25.708946,0,0
4,2025-04-19 00:00:04,-0.070246,0.209467,9.620639,111.131062,22.949712,0,0
...,...,...,...,...,...,...,...,...
995,2025-04-19 00:16:35,0.113365,0.321045,9.817748,130.569152,24.926552,1,1
996,2025-04-19 00:16:36,1.295021,-0.007956,9.835775,118.443764,27.608616,2,1
997,2025-04-19 00:16:37,0.192253,-0.264562,9.685824,93.594044,29.669365,0,0
998,2025-04-19 00:16:38,-0.171354,-0.048920,9.843418,132.867563,26.212054,0,0


### Limpieza

In [9]:
df = df.dropna()

### Variables de entrada y variable objetivo

In [15]:
X = df[["Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)", "Strain (με)", "Temp (°C)"]]
y = df["Damage"]

### División train/test (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

### Crear y entrenar el modelo

In [ ]:
modelo = Pipeline([("scaler", StandardScaler()),("logreg", LogisticRegression(max_iter=1000))])

modelo.fit(X_train, y_train)


Modelo entrenado correctamente


In [19]:
accuracy = modelo.score(X_test, y_test)
print(accuracy)

0.8342541436464088


### Sacamos los pesos que ha dado el modelo a nuestras columnas

In [21]:
coeficientes = modelo.named_steps["logreg"].coef_[0]

resultado = pd.DataFrame({
    "Sensor": X.columns,
    "Coeficiente": coeficientes,
    "Peso_absoluto": np.abs(coeficientes)
})

resultado["Peso_normalizado"] = (
    resultado["Peso_absoluto"] / resultado["Peso_absoluto"].sum()
)

resultado = resultado.sort_values("Peso_normalizado", ascending=False)

print(resultado)

            Sensor  Coeficiente  Peso_absoluto  Peso_normalizado
3      Strain (με)     2.125725       2.125725          0.655059
0  Accel_X (m/s^2)     0.794993       0.794993          0.244984
1  Accel_Y (m/s^2)     0.242794       0.242794          0.074819
4        Temp (°C)     0.044995       0.044995          0.013865
2  Accel_Z (m/s^2)     0.036580       0.036580          0.011272


Con esto, ahora damos pesos a nuestros sensores:

Strain $\rightarrow$ Piezo $\rightarrow 65.50%$

Temperatura $\rightarrow 1.38%$

Como, dependiendo de el caso específico de la medición puede ser un componente de la aceleración más importante que otro, y las mediciones no son expresamente para el mismo caso que el nuestro, lo más seguro es juntar las tres componentes y repartir el peso total entre las 3.

Accel_X + Accel_Y + Accel_Z = Accel

24.49 + 7.48 + 1.13 = 33.1

Accel $\rightarrow 33.1%$

In [28]:
peso_acc = resultado.loc[resultado["Sensor"].isin(["Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)"]),"Peso_normalizado"].sum()

print(peso_acc)

0.3310750770437362


In [29]:
peso_acc = resultado.loc[
    resultado["Sensor"].isin(["Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)"]),
    "Peso_normalizado"
].sum()

peso_piezo = resultado.loc[
    resultado["Sensor"] == "Strain (με)",
    "Peso_normalizado"
].iloc[0]

peso_temp = resultado.loc[
    resultado["Sensor"] == "Temp (°C)",
    "Peso_normalizado"
].iloc[0]

print("Peso acelerómetro:", peso_acc)
print("Peso piezo:", peso_piezo)
print("Peso temperatura:", peso_temp)

Peso acelerómetro: 0.3310750770437362
Peso piezo: 0.6550594802630247
Peso temperatura: 0.013865442693238925
